In [ ]:
# If Basemap is missing, uncomment and run this cell once:
# %pip install basemap


In [ ]:
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.basemap import Basemap
import requests
from scipy import stats

# Read data locally when available, or from GitHub in Colab.
DATA_URL = 'https://raw.githubusercontent.com/alexeykosh/intro-to-ling-2026-S3/main/S9/'
def data_path(filename):
    for folder in (Path('S9'), Path('.')):
        if (folder / filename).exists():
            return folder / filename
    return DATA_URL + filename

sns.set(context='paper', style='ticks', font_scale=1, palette='colorblind')


## 0. Question

Today, we will answer the following question:

*Does the average temperature affect the colexification of SNOW and ICE in the languages of the world?*

Yesterday, we looked at [this paper](https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0151138), where the authors tested whether languages spoken in the colder climates are more likely to colexify the senses ICE and SNOW than the languages spoken in the warmer climates. However, they have only looked at 166 languages, while we have the possibility to test this hypothesis on a much larger set of languages from the [Lexibank](https://lexibank.clld.org/) database. The goal of today session would be to reproduce their results on a much larger sample of languages

## 1. Data

### 1.1. Extracting data from Lexibank

First, we will need to extract data from Lexibank. It has been conveniently gathered for you by the creators of this database, so you can look at it below:

In [ ]:
all_lexibank = pd.read_csv(data_path('lexibank_clipped.csv.gz'),
                           low_memory=False, index_col=0)
all_lexibank.shape


Let's try to make sense of different columns first:

In [ ]:
all_lexibank.columns

What columns do you think we will need to consider for testing the hypothesis in question? 

In [ ]:
##################
# YOUR CODE HERE #
##################

How many unique languages are there?

In [ ]:
##################
# YOUR CODE HERE #
##################

How many linguistc families?

In [ ]:
##################
# YOUR CODE HERE #
##################

Can you find data on French? Does the data make sense for you?

In [ ]:
##################
# YOUR CODE HERE #
##################

Can you exclude languages that do not have a family assigned to them?

In [ ]:
##################
# YOUR CODE HERE #
##################

Ok, now let's focus on our hypothesis. We are interested in two senses from lexibank, namely SNOW and ICE. Let's store them in a list as strings:

In [ ]:
terms_for_snow = [
    "ICE", "SNOW"
]

We will not need all the columns, but we will need the following four:

In [ ]:
all_lexibank = all_lexibank[['Form', 'Concepticon_Gloss', 'Glottocode', 'Family']]

Get the subset of the forms from Lexibank that contain words for senses SNOW and ICE. Save it in the variable `subset_si`.

In [ ]:
##################
# YOUR CODE HERE #
##################

In [ ]:
subset_si.head()


In [ ]:
subset_si.shape

Great, now we have the subset of words

In [ ]:
subset_si.head(5)

Let's look at a language that has two separate forms:

In [ ]:
subset_si.query('Glottocode == "russ1263"')

And just one form for both:

In [ ]:
subset_si.query('Glottocode == "hawa1245"')

Now you need to extract the number of unique terms (unique Form) for every language in the database. **Note that we only need to consider languages that have at least two Concepticon_Gloss entries.** Write your code below to fill the list `lang_colex` with the following entries `[glottocode, number_of_unique_forms]`

In [ ]:
##################
# YOUR CODE HERE #
##################

Let's look at the results. How many languages do we have?

In [ ]:
lang_colex[:5]

In [ ]:
len(lang_colex)

Now let's convert it into a dataframe:

In [ ]:
##################
# YOUR CODE HERE #
##################

Let's look at the distibution of the number of terms in the df_colex dataframe:

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df_colex['Number_of_terms'],
         bins=range(1, 7),
         align='left')
sns.despine()
plt.show()

Let's look at the languages that have more than two terms. Is there something interesting that you can notice?

In [ ]:
##################
# YOUR CODE HERE #
##################

Normally, the language with the highest number of terms should be Romanian, so let's look a the terms in Romanian -- does the data look good?

In [ ]:
subset_si.query('Glottocode == "roma1327"')

Finally, we need latitude and longitude from Glottolog. Let’s load the data first:


In [ ]:
glottolog = pd.read_csv(data_path('glottolog.csv'))


In [ ]:
glottolog.head(5)

Merge `df_colex` with `glottolog` on `glottocode` using a left merge, and save the result as `df_merged`.


In [ ]:
##################
# YOUR CODE HERE #
##################

Check the shape:

In [ ]:
df_merged.shape

Now we need to remove languages that do not have coordinates:

In [ ]:
##################
# YOUR CODE HERE #
##################

Check the results again:

In [ ]:
df_merged.shape

### 1.2. Getting average temperatures at each language location

We have latitude and longitude for each language. The function below shows how the NASA POWER API was used to calculate average daily temperatures from 2020 to 2025. **Do not run it for every language in class:** the results are already saved in `temperature_data.csv` and loaded in the next section.


In [ ]:
def get_overall_avg_temp(lat, lon, start_year, end_year):
    """
    Fetches overall average temperature for a given latitude and longitude using the NASA POWER API.
    Returns a DataFrame with the overall average for the range.
    """
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {
        "parameters": "T2M",
        "community": "AG",
        "longitude": lon,
        "latitude": lat,
        "start": f"{start_year}0101",
        "end": f"{end_year}1231",
        "format": "JSON",
    }
    response = requests.get(url, params=params)
    if response.status_code != 200:
        print(f"Failed to fetch data for {lat}, {lon}: Status {response.status_code}")
        return None

    # Parse daily temperatures
    data = response.json()['properties']['parameter']['T2M']
    df_temp = pd.DataFrame(list(data.items()), columns=["Date", "Temperature"])
    df_temp["Date"] = pd.to_datetime(df_temp["Date"])
    df_temp.set_index("Date", inplace=True)

    # Calculate overall average temperature for the specified range
    overall_avg = df_temp["Temperature"].mean()
    return pd.DataFrame({
        "latitude": [lat],
        "longitude": [lon],
        "overall_avg_temp": [overall_avg]
    })

# Function to apply to DataFrame
def apply_overall_avg_temp(df, start_year, end_year):
    """
    Applies `get_overall_avg_temp` to each row in a DataFrame with 'latitude' and 'longitude' columns.
    Returns a DataFrame with overall average temperatures for each location.
    """
    results = []
    for _, row in tqdm(df.iterrows()):
        overall_data = get_overall_avg_temp(row['latitude'], row['longitude'], start_year, end_year)
        if overall_data is not None:
            results.append(overall_data)
    return pd.concat(results, ignore_index=True)

# start_year, end_year = 2020, 2025
# df_results = apply_overall_avg_temp(df_merged, start_year, end_year)
# df_results.to_csv('temperature_data.csv')

Let's load the data:

In [ ]:
df_results = pd.read_csv(data_path('temperature_data.csv'), index_col=0)
# The saved table repeats one location; keep one temperature per location.
df_results = df_results.drop_duplicates(subset=['latitude', 'longitude'])

Let's look at the data:

In [ ]:
df_results.head()

What's the max temperature in this dataset? And the minimum?

In [ ]:
df_results.describe()

Merge `df_merged` with `df_results` on latitude and longitude, using a right merge. Save it as `df_merged_f`.


In [ ]:
##################
# YOUR CODE HERE #
##################

Create `Number_of_terms_bin`: `True` if there are at least two distinct forms for SNOW and ICE, and `False` if there is only one. `False` thus means colexification.


In [ ]:
##################
# YOUR CODE HERE #
##################

We should have data on temperature and colexification for 241 languages. Check the number of unique glottocodes, then inspect the data. Do you see any outliers?


In [ ]:
df_merged_f.glottocode.nunique()

In [ ]:
df_merged_f.head()

## 2. Analysis

Let's reproduce the two boxplots comparing average temperature where SNOW and ICE share a form (`False`) and where they have different forms (`True`). 

Did we manage to reproduce the results from the original paper?

In [ ]:
plot_data = df_merged_f.assign(
    Terms_for_snow_and_ice=df_merged_f['Number_of_terms_bin'].map({False: 'same', True: 'different'})
)
plt.figure(figsize=(8, 4))
plt.axvline(x=0, color='red', linestyle='--', linewidth=3)
sns.boxplot(data=plot_data, x='overall_avg_temp',
            y='Terms_for_snow_and_ice', fill=False)
plt.xlabel('Average temperature')
plt.ylabel('Terms for snow and ice')
plt.show()


Perform a Welch t-test using `stats.ttest_ind` on the temperature values in the two groups. Set `equal_var=False`.


In [ ]:
##################
# YOUR CODE HERE #
##################

How do you interpret the results? Do they support the hypothesis that languages spoken in colder climates are more likely to colexify SNOW and ICE?

Now let's recreate a map showing the locations of languages with the same and different forms for SNOW and ICE. Use `Basemap` with `projection="cyl"` to draw coastlines, then plot longitude on the x axis and latitude on the y axis.


In [ ]:
# Prepare data
x_same = df_merged_f.loc[~df_merged_f['Number_of_terms_bin'], 'longitude']
y_same = df_merged_f.loc[~df_merged_f['Number_of_terms_bin'], 'latitude']
x_different = df_merged_f.loc[df_merged_f['Number_of_terms_bin'], 'longitude']
y_different = df_merged_f.loc[df_merged_f['Number_of_terms_bin'], 'latitude']

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# First subplot for the same form
ax = axes[0]
m1 = Basemap(projection='cyl', resolution='c', ax=ax)
m1.drawcoastlines(color='black')
m1.fillcontinents(color='white', lake_color='white')
m1.plot(x_same, y_same, 'ro', markersize=3, alpha=1)
ax.set_title('Same Term for SNOW and ICE', fontsize=10, weight='bold')

# Second subplot for different forms
ax = axes[1]
m2 = Basemap(projection='cyl', resolution='c', ax=ax)
m2.drawcoastlines(color='black')
m2.fillcontinents(color='white', lake_color='white')
m2.plot(x_different, y_different, 'bo', markersize=3, alpha=1)
ax.set_title('Different Terms for SNOW and ICE', fontsize=10, weight='bold')

plt.tight_layout()
plt.show()

## 3. Additional analyses

### 3.1. Controlling for geographical location

The colexifying languages (`Number_of_terms_bin == False`) cover a limited **latitude** range. Find their minimum and maximum latitude and store them in `min_lat` and `max_lat`.


In [ ]:
##################
# YOUR CODE HERE #
##################


Restrict both groups to latitudes between `min_lat` and `max_lat`, and repeat the boxplot. What changes? This simple restriction does not control for language family or other geographical differences.


In [ ]:
##################
# YOUR CODE HERE #
##################


In [ ]:
##################
# YOUR CODE HERE #
##################


Repeat the Welch t-test on the latitude-restricted sample. How does the result compare with the original test?


In [ ]:
##################
# YOUR CODE HERE #
##################
